# Lab 1 — Time series exploratory data analysis (EDA)

**Goal:** build intuition from plots, quantify serial dependence (ACF/PACF), and decompose a series into interpretable parts before modelling.

**Prerequisite:** run `generate_data.ipynb` so `data/synthetic_series.csv` exists.

Theory excerpts below align with `time_series_forecasting.md` (forecasting perspective, graphics, autocorrelation, decomposition).

## 1. Why plot first?

> The first thing to do in any data analysis task is to plot the data. Graphs enable many features of the data to be visualised, including patterns, unusual observations, changes over time, and relationships between variables.

For forecasting, you want to see **level**, **trend**, **seasonality**, **cycles** (non-fixed period), **heteroskedasticity** (variance changing with level), and **structural breaks** or outliers. The features you see should guide preprocessing (transformations), season length, and model family.

## 2. Pattern vocabulary

- **Trend** — sustained increase or decrease over time; need not be linear.
- **Seasonality** — variation tied to a **fixed, known period** (e.g. day-of-week, month).
- **Cycle** — rises and falls **without** fixed period (often macro / business cycle). Seasonal ≠ cyclic: seasonality repeats on a calendar; cycles do not.

**Forecasting framing:** quantitative methods assume some aspects of past behaviour continue. Pure extrapolation methods use only the target series; explanatory models add covariates; mixed models combine lags and regressors.

## 3. Autocorrelation and "memory"

**Autocorrelation** measures linear association between $y_t$ and $y_{t-k}$:

$$
r_k = \frac{\sum_{t=k+1}^{T}(y_t-\bar y)(y_{t-k}-\bar y)}{\sum_{t=1}^{T}(y_t-\bar y)^2}
$$

The collection $\{r_k\}$ is the **ACF**. For **trended** series, ACF decays slowly with positive values; for **seasonal** data, spikes often appear at seasonal lags; **white noise** has ACF near zero at all lags (aside from sampling noise).

Rough **95% confidence bounds** for white noise: $\pm 1.96/\sqrt{T}$ (often shown as horizontal bands on ACF plots).

## 4. Decomposition idea

A series is often split into **trend–cycle**, **seasonal**, and **remainder**:

**Additive:** $y_t = T_t + S_t + R_t$ — seasonal swings have roughly constant *absolute* size.

**Multiplicative:** $y_t = T_t \cdot S_t \cdot R_t$ — swings scale with level (common in economics). Equivalently, $\log y_t = \log T_t + \log S_t + \log R_t$ gives an additive decomposition on the log scale.

**STL** (Seasonal and Trend decomposition using Loess) is flexible: arbitrary season length, seasonal pattern can evolve, robust to outliers (optional). It is **additive** on the given scale — if variability is multiplicative, consider a log transform first.

**Classical** decomposition is simpler but assumes **fixed** seasonal shape and can miss endpoints / sharp turns.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL, seasonal_decompose
from statsmodels.tsa.stattools import adfuller

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 3.5)

DATA_PATH = Path("data/synthetic_series.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing {DATA_PATH}. Run generate_data.ipynb first."
    )

raw = pd.read_csv(DATA_PATH, parse_dates=["date"])
raw = raw.sort_values("date").reset_index(drop=True)

# Prefer meta written by generate_data.ipynb; fallback matches default generator
_meta_path = Path("data/synthetic_series_meta.json")
if _meta_path.exists():
    import json

    with open(_meta_path, encoding="utf-8") as f:
        _meta = json.load(f)
    SEASONAL_PERIOD = int(_meta.get("seasonal_period", 7))
else:
    SEASONAL_PERIOD = 7

df = raw.set_index("date").asfreq("D")  # regular daily grid; fills if any gaps
y = df["y"].astype(float)

print(y.describe())
print("N =", len(y), "  freq =", y.index.inferred_freq)

### Time plot

Join consecutive observations in time order. Look for level shifts, outliers, and whether variability grows with the mean (suggests log or Box–Cox).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y.index, y.values, lw=0.8)
ax.set_title("Time plot: $y_t$")
ax.set_xlabel("date")
plt.tight_layout()
plt.show()

### Rolling statistics (local level & volatility)

A moving average smooths short-term fluctuations so **slow trend** is easier to see. Compare several window lengths relative to `SEASONAL_PERIOD`.

In [ ]:
win_short = SEASONAL_PERIOD
win_long = 4 * SEASONAL_PERIOD

roll_mean_s = y.rolling(win_short, center=True).mean()
roll_mean_l = y.rolling(win_long, center=True).mean()
roll_std = y.rolling(win_long, center=True).std()

fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
axes[0].plot(y.index, y.values, lw=0.6, alpha=0.6, label="y")
axes[0].plot(roll_mean_s.index, roll_mean_s.values, lw=1.2, label=f"MA({win_short})")
axes[0].plot(roll_mean_l.index, roll_mean_l.values, lw=1.4, label=f"MA({win_long})")
axes[0].set_title("Series with moving-average smoothers")
axes[0].legend(loc="upper left", ncol=3, fontsize=8)

axes[1].plot(roll_std.index, roll_std.values, color="C3", lw=1)
axes[1].set_title(f"Rolling std (window={win_long})")
plt.tight_layout()
plt.show()

### Distribution and normal QQ (sanity)

Time series are **not i.i.d.**; still, inspecting the marginal distribution helps detect heavy tails, skew, or multi-modality (often from mixing regimes).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(y, kde=True, ax=axes[0], bins=40)
axes[0].set_title("Histogram of $y_t$")

sns.scatterplot(x=y.index.dayofyear, y=y.values, ax=axes[1], s=8, alpha=0.35)
axes[1].set_xlabel("day of year")
axes[1].set_ylabel("y")
axes[1].set_title("Seasonal cloud: y vs day-of-year")
plt.tight_layout()
plt.show()

### Seasonal plot & weekday subseries

For `SEASONAL_PERIOD = 7`, overlay each week on the seasonal axis or compare **by weekday** to see stable day-of-week effects.

In [ ]:
s = y.copy()
s.name = "y"
wk = pd.DataFrame({"y": s})
wk["dow"] = wk.index.dayofweek  # Mon=0 ... Sun=6
wk["week"] = (np.arange(len(wk)) // SEASONAL_PERIOD).astype(int)

fig, ax = plt.subplots(figsize=(11, 4))
for w in range(min(20, wk["week"].max() + 1)):
    seg = wk.loc[wk["week"] == w, "y"].reset_index(drop=True)
    ax.plot(seg.index, seg.values, lw=0.9, alpha=0.35)
ax.set_xlabel("index within 7-day block")
ax.set_ylabel("y")
ax.set_title("Seasonal plot: first 20 weeks overlaid")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
data_box = [wk.loc[wk["dow"] == d, "y"].values for d in range(7)]
ax.boxplot(data_box, labels=order, showfliers=False)
ax.set_title("Distribution of $y$ by weekday")
plt.tight_layout()
plt.show()

### Lag scatter (visual ACF)

Plot $y_t$ vs $y_{t-k}$ for small $k$. Curved structure means linear ACF alone is insufficient (nonlinear dynamics).

In [ ]:
lags = [1, SEASONAL_PERIOD, SEASONAL_PERIOD + 1]
fig, axes = plt.subplots(1, len(lags), figsize=(4 * len(lags), 3.5))
for ax, k in zip(axes, lags):
    pd.plotting.lag_plot(y, lag=k, ax=ax, marker=".", markersize=2, linestyle="None")
    ax.set_title(f"Lag {k}")
plt.suptitle("Lag plots", y=1.02)
plt.tight_layout()
plt.show()

### ACF and PACF

- **ACF** — correlation with past lags (combined direct + indirect effects).
- **PACF** — correlation after removing intermediate lags; useful for AR order intuition.

For series with strong trend, consider differencing or examining ACF of **detrended** or **seasonally differenced** series in later labs.

In [ ]:
max_lag = min(5 * SEASONAL_PERIOD, len(y) // 2 - 1)
fig, ax = plt.subplots(figsize=(11, 3.5))
plot_acf(y, lags=max_lag, ax=ax, title=f"ACF (lags 1..{max_lag})")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(11, 3.5))
plot_pacf(y, lags=max_lag, ax=ax, method="ywm", title=f"PACF (lags 1..{max_lag})")
plt.tight_layout()
plt.show()

### Quick stationarity check (Augmented Dickey–Fuller)

Many classical models assume **stationarity after differencing**. ADF tests the null of a unit root (non-stationary). Interpret with caution: power is limited on short samples.

In [ ]:
adf_stat, adf_p, *_ = adfuller(y.dropna(), autolag="AIC")
print(f"ADF statistic: {adf_stat:.4f}")
print(f"p-value: {adf_p:.4g}")

### Classical decomposition (`seasonal_decompose`)

Compare **additive** vs **multiplicative** on this synthetic series. If variance grows with level, multiplicative (or log + additive STL) is often more natural.

In [ ]:
dec_add = seasonal_decompose(y, model="additive", period=SEASONAL_PERIOD)
dec_mul = seasonal_decompose(y, model="multiplicative", period=SEASONAL_PERIOD)

fig = dec_add.plot()
fig.set_size_inches(10, 8)
plt.suptitle("Classical additive decomposition", y=1.02)
plt.tight_layout()
plt.show()

fig = dec_mul.plot()
fig.set_size_inches(10, 8)
plt.suptitle("Classical multiplicative decomposition", y=1.02)
plt.tight_layout()
plt.show()

### STL decomposition (robust, evolving seasonality)

Tune `seasonal` and `trend` window sizes (odd integers). Defaults are often fine; increase seasonal window if the pattern is noisy, decrease if seasonality changes quickly.

In [ ]:
stl = STL(y, seasonal=2 * SEASONAL_PERIOD + 1, robust=True)
res = stl.fit()

fig = res.plot()
fig.set_size_inches(10, 8)
plt.suptitle("STL (robust)", y=1.02)
plt.tight_layout()
plt.show()

remainder = res.resid
fig, ax = plt.subplots(figsize=(11, 3))
plot_acf(remainder.dropna(), lags=min(40, len(remainder) // 2 - 1), ax=ax)
ax.set_title("ACF of STL remainder (should look closer to noise if structure captured)")
plt.tight_layout()
plt.show()

### Optional: compare to known components (teaching only)

Our generator saved latent `trend`, `seasonal`, etc. **Real data do not provide these.** Use this cell only to build intuition about what STL is trying to recover.

In [ ]:
if {"trend", "seasonal"}.issubset(df.columns):
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(df.index, df["trend"], label="true trend", lw=1.2)
    ax.plot(res.trend.index, res.trend, label="STL trend", lw=1, alpha=0.85)
    ax.legend()
    ax.set_title("Trend: simulated vs STL estimate")
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(11, 3.5))
    ax.plot(df.index, df["seasonal"], label="true seasonal", lw=1)
    ax.plot(res.seasonal.index, res.seasonal, label="STL seasonal", lw=0.9, alpha=0.85)
    ax.legend()
    ax.set_title("Seasonal: simulated vs STL estimate")
    plt.tight_layout()
    plt.show()
else:
    print("No latent columns in CSV — skip comparison.")

## Interview-style checklist (what to mention)

1. **Identify** trend vs seasonal vs cycle; note whether variation scales with level.
2. **Choose scale** — raw vs log; match decomposition (additive on that scale).
3. **ACF/PACF** — long-memory trend vs seasonal spikes; inform differencing / SARIMA orders.
4. **Decomposition goal** — isolate stable seasonal pattern and smooth trend for reporting; **remainder** should look like noise (or known covariates explain the rest).
5. **Honest limits** — EDA does not replace cross-validation on rolling forecast origins.

**Next labs:** engineered features (lags, calendars), baselines, SARIMA/SARIMAX, gradient boosting with lag features, sequence models.